# overnight — 전체 실행 한 방 (diag → seed 학습 → eval → 집계)

**Run All 걸어 놓고 자면 되는 노트북이다.** 위에서 아래로 한 번에 돈다.

## 프로토콜 — 논문 ablation 표와 같은 조건

| | |
|---|---|
| 학습 | **100k step** (`common_final` 기본값 150k 를 이 노트북이 덮어쓴다) |
| eval 체크포인트 | **100k** (정확히 그 step 만. 없으면 그 셀은 eval 하지 않는다) |
| eval 에피소드 | **50 ep / task × 10 task = overall 500** |
| 조건 | K=100, 실행 stride s=10, TE off |

**사전 점검(§2)이 위 값이 실제 커맨드에 들어갔는지 확인하고, 틀리면 학습 전에 중단한다.**
(`common_final` 은 150k 로 고정돼 있어서 덮어쓰지 않으면 조용히 150k 를 돈다.)

| 단계 | 무엇 | GPU | 시간 |
|---|---|---|---|
| **A** | diag0/1/2/3 를 **100k 체크포인트로** 다시 (학습 없음) | 0 하나 | ~20분 |
| **B** | seed 1·2 학습 — `bimamba_pure`, `acm2` × K=100 = **4잡** | 0~3 | **~5.5h** |
| **C** | eval — seed 별 2셀 (500 ep) | 0~1 | ~2h × 2 seed |
| **D** | 집계 — gap 평균 ± 표준편차, 요약 파일 저장 | — | 즉시 |

총 **~10h 안팎**. B 가 대부분이다.

> diag0/1/2 를 100k 로 다시 도는 이유: 앞서 잰 수치는 150k 체크포인트였는데
> 성공률 65.0 / 56.8 은 100k 다. 표현 수치와 성공률이 같은 체크포인트여야 논문에서 잇는다.

## 왜 seed 재현인가

논문 헤드라인 `BiMamba 65.0 vs ACM2 56.8 = +8.2` 가 **seed 0 하나**다. 리뷰어가 제일
먼저 묻는 곳이고, diag 로 메커니즘을 규명해도 이게 안 받쳐 주면 소용없다.
**코드 작업이 0 이고 잃을 게 없는 유일한 실험**이라 제일 먼저 건다.

## 무인 실행 안전장치

- **사전 점검(§2)이 틀리면 즉시 중단한다.** 순수 BiMamba 가 `--use_chunk_pairs`
  를 달고 있거나 `--policy.sscp_enabled=false` 가 없으면 8시간을 날리므로 **여기서만 멈춘다.**
- **그 뒤 단계는 서로 격리돼 있다.** A 가 죽어도 B 는 돈다. B 가 중간에 죽어도 C·D 는
  가진 것만으로 진행한다.
- **각 단계가 끝날 때마다 상태를 파일로 쓴다** (`outputs/final/share/overnight/`).
  브라우저를 닫으면 셀 출력이 사라질 수 있으니 **아침에는 이 파일부터 본다.**
- **다시 실행해도 안전하다.** 100k 인 학습은 skip, 중단된 건 resume, 끝난 eval 은 skip.

> ⚠️ 이 노트북은 `exp5_tonight.py` 와 `diag3_bk_variants.py` 를 부르기만 한다.
> 잡 정의·스킵·집계는 전부 거기 있다.

## 0) 부팅

In [ ]:
import json, os, re, subprocess, sys, time, traceback
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
REPO = _r

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

# ── 프로토콜: 100k 학습 / 100k eval ─────────────────────────────────────────────
# common_final.py:72-74 가 v23.STEPS = CKPT_STEP = 150_000 으로 고정한다.
# setup() 이 common_final 을 reload 하므로 **반드시 setup() 뒤에** 덮어써야 하고,
# 단계마다 다시 호출해서 어디선가 reload 돼도 100k 가 유지되게 한다.
TRAIN_STEP = 100_000

def apply_protocol():
    v23.STEPS = TRAIN_STEP       # make_train_cmd 가 --steps 로 읽는다
    cf.STEPS = TRAIN_STEP
    cf.CKPT_STEP = TRAIN_STEP    # run_training_jobs 의 skip 기준 + run_evals 의 select
apply_protocol()

STAMP = time.strftime('%Y%m%d_%H%M')
SHARE = Path(os.environ.get('LEROBOT_OUTPUT',
                            Path.home() / 'lerobot_project' / 'outputs')) / 'final' / 'share' / 'overnight'
SHARE.mkdir(parents=True, exist_ok=True)
STATUS_PATH = SHARE / f'status_{STAMP}.json'
STATUS = {'stamp': STAMP, 'stages': {}}

def save_status():
    STATUS_PATH.write_text(json.dumps(STATUS, ensure_ascii=False, indent=1, default=str),
                           encoding='utf-8')

def stage(name, fn):
    """단계를 격리해서 돌린다. 죽어도 다음 단계는 계속 간다."""
    t0 = time.time()
    print('\n' + '=' * 70 + f'\n[{name}] 시작 {time.strftime("%H:%M:%S")}\n' + '=' * 70)
    try:
        out = fn()
        STATUS['stages'][name] = {'ok': True, 'sec': round(time.time() - t0)}
        return out
    except Exception as e:
        traceback.print_exc()
        print(f'\n!! [{name}] 실패 — {type(e).__name__}: {e}')
        print('   이 단계만 건너뛰고 다음으로 간다.')
        STATUS['stages'][name] = {'ok': False, 'sec': round(time.time() - t0),
                                  'error': f'{type(e).__name__}: {e}'}
        return None
    finally:
        save_status()
        print(f'[{name}] 끝 — {STATUS["stages"][name]}')

print('repo      :', REPO)
print('TASK      :', X.TASK, '  기본 SEED:', X.SEED)
print('MAIN_STRIDE:', X.MAIN_STRIDE, ' N_EP:', X.N_EP, '(task 당 -> overall 500)')
print('학습 step :', f'{v23.STEPS:,}', '  eval ckpt:', f'{cf.CKPT_STEP:,}')
print('GPU       :', v23.available_gpus())
print('상태 파일 :', STATUS_PATH)

## 1) 설정

In [ ]:
GPUS     = [0, 1, 2, 3]            # GPU 4개
K        = 100                     # 헤드라인 셀
SEEDS    = [1, 2]                  # 새로 돌릴 seed (0 은 이미 있음)
VARIANTS = ['bimamba', 'acm2']     # exp5 변형 이름. bimamba -> tag 'bimamba_pure'

DIAG_GPU   = '0'                   # diag3 은 학습 시작 전에 끝나므로 아무 GPU 나 된다
DIAG_SEED  = 0                     # diag3 이 여는 체크포인트 seed
DIAG_STEP  = TRAIN_STEP            # 성공률과 같은 체크포인트(100k)로 잰다

TAGS = [X.tag_of(v, K) for v in VARIANTS]
JOBS = [(t, s, X.TASK) for s in SEEDS for t in TAGS]

print('태그 :', TAGS)
print('잡   :', len(JOBS), '개')
for j in JOBS:
    print('   ', j)
STATUS['config'] = {'K': K, 'seeds': SEEDS, 'variants': VARIANTS, 'jobs': [list(j) for j in JOBS]}
save_status()

## 2) 사전 점검 — 틀리면 여기서 멈춘다

**이 셀만 일부러 예외를 던져 Run All 을 중단시킨다.** 몇 시간을 날리는 것보다 낫다.

- **`--steps=100000`** 이 학습 커맨드에 들어갔는가 (`common_final` 기본값 150k 를 덮어썼는가)
- **eval 체크포인트 = 100k**, **`N_EP` = 50** (task 당 → overall 500)
- `bimamba_pure` 커맨드에 `--use_chunk_pairs` 가 **없어야** 한다
- `bimamba_pure` 커맨드에 `--policy.sscp_enabled=false` 가 **있어야** 한다
- 모든 잡의 커맨드에 `--seed={s}` 가 **있어야** 한다
- 출력 경로에 `seed{s}` 가 **있어야** 한다 (seed 0 체크포인트를 덮어쓰지 않는지)

In [ ]:
apply_protocol()
problems = []

# 프로토콜
if cf.CKPT_STEP != TRAIN_STEP:
    problems.append(f'eval 체크포인트가 {cf.CKPT_STEP:,} 다 (기대 {TRAIN_STEP:,})')
if X.N_EP != 50:
    problems.append(f'N_EP 가 {X.N_EP} 다 (기대 50 = task 당 50ep, overall 500)')
print(f'프로토콜  학습 {v23.STEPS:,} step | eval ckpt {cf.CKPT_STEP:,} | {X.N_EP} ep/task -> overall {10 * X.N_EP}')

for t, s, task in JOBS:
    cmd = v23.make_train_cmd(t, s, task, gpu_id=GPUS[0])
    ok_seed = f'--seed={s}' in cmd
    ok_dir  = f'seed{s}' in cmd
    m = re.search(r'--steps=([0-9]+)', cmd)
    steps_in_cmd = int(m.group(1)) if m else None
    print(f'--- {t}  seed{s} ---')
    print('  --steps =', f'{steps_in_cmd:,}' if steps_in_cmd else '없음', '| seed 플래그:', ok_seed,
          '| 출력 경로 seed 표기:', ok_dir)
    if steps_in_cmd != TRAIN_STEP:
        problems.append(f'{t} seed{s}: --steps={steps_in_cmd} (기대 {TRAIN_STEP}). 150k 로 돌면 표와 안 맞는다')
    if not ok_seed:
        problems.append(f'{t} seed{s}: --seed={s} 가 커맨드에 없다')
    if not ok_dir:
        problems.append(f'{t} seed{s}: 출력 경로에 seed{s} 가 없다 (seed 0 을 덮어쓸 수 있다)')
    if t.startswith('bimamba_pure'):
        cp = '--use_chunk_pairs' in cmd
        sf = '--policy.sscp_enabled=false' in cmd
        print('  chunk_pairs 있음:', cp, '(False 여야) | sscp_enabled=false 있음:', sf, '(True 여야)')
        if cp:
            problems.append(f'{t} seed{s}: 순수 BiMamba 인데 --use_chunk_pairs 가 있다')
        if not sf:
            problems.append(f'{t} seed{s}: 순수 BiMamba 인데 --policy.sscp_enabled=false 가 없다')

STATUS['preflight'] = {'ok': not problems, 'problems': problems,
                       'train_step': v23.STEPS, 'eval_ckpt': cf.CKPT_STEP, 'n_ep_per_task': X.N_EP}
save_status()
if problems:
    raise AssertionError('사전 점검 실패 — 학습을 시작하지 않는다:\n  ' + '\n  '.join(problems))
print('\n사전 점검 통과.')

## 3) 단계 A — diag0 / 1 / 2 / 3 를 100k 체크포인트로

앞서 잰 diag0/1/2 는 **150k** 체크포인트였고 성공률 `65.0 / 56.8` 은 **100k** 다.
**표현 수치와 성공률이 같은 체크포인트**여야 논문에서 잇는다. 학습 없이 forward pass 뿐이라
셋 다 다시 돌려도 ~20분이고, **학습 시작 전에 끝나서 GPU 를 안 뺏는다.**

| | 무엇 | 답 |
|---|---|---|
| diag0 | 역방향 브랜치가 상수인가 | `BWD @ query` = 0 |
| diag1 | `b_k` 표로 바꿔도 출력이 같은가 | `max\|Δ\|` = 0 (항등식) |
| diag2 | 순방향은 위치가 뭉개지고 `b_k` 가 갈라 주는가 | 위치 활용률 `fwd` vs `fused` |
| diag3 | `b_k` 의 어떤 성질이 일하는가 | `mean` 회복률 (낮으면 위치별 변화가 원천) |

**한 스크립트가 죽어도 다음이 계속 간다.** 100k 체크포인트가 없는 태그는 그 태그만 건너뛴다.

In [ ]:
DIAG_DIR = REPO / 'notebooks' / 'libero'
DIAG_ENV = dict(os.environ,
                PYTHONPATH=str(REPO / 'src'),
                HF_HUB_DISABLE_XET='1',
                MPLBACKEND='Agg',
                CUDA_VISIBLE_DEVICES=DIAG_GPU)

# 스크립트별 추가 인자. --tags all --seed --step --task --batch --json 은 공통이다.
DIAG_RUNS = {
    'diag0': ('diag0_backward_const.py', []),
    'diag1': ('diag1_bk_identity.py', ['--save-dir', str(SHARE / 'diag1')]),
    'diag2': ('diag2_position_collapse.py', ['--acm2', '--errors', '--save-dir', str(SHARE / 'diag2')]),
    'diag3': ('diag3_bk_variants.py', []),
}
DIAG_RES = {}

def run_diag(name):
    script, extra = DIAG_RUNS[name]
    json_path = SHARE / f'{name}_all_{STAMP}_{DIAG_STEP // 1000}k.json'
    cmd = [v23.PYTHON, str(DIAG_DIR / script), '--tags', 'all', '--seed', str(DIAG_SEED),
           '--step', str(DIAG_STEP), '--task', X.TASK, '--batch', '4',
           '--json', str(json_path)] + extra
    print('$', ' '.join(cmd), '\n')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=DIAG_ENV, cwd=str(REPO))
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    print(f'\n[exit {rc}]')
    res = json.loads(json_path.read_text(encoding='utf-8')) if json_path.exists() else {}
    STATUS.setdefault('diag', {})[name] = {'exit': rc, 'json': str(json_path), 'step': DIAG_STEP}
    # 서브프로세스가 죽어도 예외가 안 나므로 직접 실패로 올린다 (상태 파일이 OK 로 거짓말하지 않게).
    # 종료 코드 0/2 는 정상 종료다 (2 = 일부 태그가 체크포인트 없음 등으로 건너뜀).
    if rc not in (0, 2) or not res:
        raise RuntimeError(f'{name} 이 결과 없이 끝났다 (exit {rc}). 위 로그의 Traceback 을 볼 것.')
    bad = [t for t, r in res.items() if not r.get('ok')]
    if bad:
        print(f'!! {name}: 일부 태그 건너뜀/실패: {bad}')
    return res

for _name in DIAG_RUNS:
    DIAG_RES[_name] = stage(f'A: {_name}', lambda n=_name: run_diag(n)) or {}

In [ ]:
# diag 요약 — 100k 체크포인트 기준. 아침에 이 표만 보면 된다.
def _diag_summary():
    summ = {}
    K_ORDER = lambda kv: kv[1].get('K', 0)

    print('[diag0] 역방향 브랜치는 상수인가')
    for tag, r in sorted(DIAG_RES.get('diag0', {}).items(), key=K_ORDER):
        if 'bwd_ab' not in r:
            continue
        rat = r.get('ratio')
        rat_m = (sum(rat) / len(rat)) if rat else float('nan')
        print(f"   {tag:<20} K={r.get('K', '?'):>3}  {r.get('verdict', '?'):<12} "
              f"BWD A-B={r['bwd_ab']:.1e}  ||bwd||/||fwd||={rat_m:.2f}")
        summ.setdefault('diag0', {})[tag] = {'verdict': r.get('verdict'), 'bwd_ab': r['bwd_ab']}

    print('\n[diag1] b_k 표로 바꿔도 출력이 같은가')
    for tag, r in sorted(DIAG_RES.get('diag1', {}).items(), key=K_ORDER):
        if 'identity_max_diff_seen' not in r:
            continue
        print(f"   {tag:<20} K={r['K']:>3}  {r['verdict']:<12} "
              f"max|Δ| seen={r['identity_max_diff_seen']:.1e} unseen={r['identity_max_diff_unseen']:.1e}  "
              f"역방향 params {r['n_params_backward']:,} -> 표 {r['n_params_table']:,}")
        summ.setdefault('diag1', {})[tag] = {'verdict': r['verdict']}

    print('\n[diag2] 위치 활용률 (eff_rank / K) — 높을수록 위치가 잘 갈라짐')
    print(f"   {'tag':<20} {'K':>4} {'fwd':>6} {'b_k off':>8} {'b_k on':>7}   오차 off -> on")
    for tag, r in sorted(DIAG_RES.get('diag2', {}).items(), key=K_ORDER):
        if not r.get('ok'):
            continue
        sp = r['sep']
        g = lambda n: f"{sp[n]['use_ratio']:.2f}" if n in sp else '  --'
        e = r.get('errors', {})
        et = (f"{e['err_off_mean']:.3f} -> {e['err_on_mean']:.3f}" if e.get('ok') else '--')
        print(f"   {tag:<20} {r['K']:>4} {g('fwd'):>6} {g('ablated'):>8} {g('fused'):>7}   {et}")
        summ.setdefault('diag2', {})[tag] = {n: sp[n]['use_ratio'] for n in sp}

    print('\n[diag3] 회복률 (zero=0%, real=100%) — mean 이 낮을수록 위치별 변화가 이득의 원천')
    ok = [(t, r) for t, r in DIAG_RES.get('diag3', {}).items() if r.get('ok')]
    if ok:
        print(f"   {'tag':<20} {'K':>4} {'mean':>7} {'shuffle':>9} {'rand_ortho':>11} {'α=0.5':>7} {'α=2':>6}")
        for tag, r in sorted(ok, key=K_ORDER):
            v = r['variants']
            span = v['zero']['err_mean'] - v['real']['err_mean']
            rec = lambda k: ((v['zero']['err_mean'] - v[k]['err_mean']) / span
                             if (k in v and span > 1e-9) else float('nan'))
            row = {k: rec(k) for k in ('mean', 'shuffle', 'random_ortho', 'alpha=0.5', 'alpha=2')}
            summ.setdefault('diag3', {})[tag] = {'K': r['K'], **row}
            print(f"   {tag:<20} {r['K']:>4} {row['mean']:>7.0%} {row['shuffle']:>9.0%} "
                  f"{row['random_ortho']:>11.0%} {row['alpha=0.5']:>7.0%} {row['alpha=2']:>6.0%}")
        ms = [x['mean'] for x in summ['diag3'].values()]
        if ms and max(ms) < 0.25:
            print('   >> mean 회복률이 전 K 에서 25% 미만 — 위치 분리 해석이 굳는다.')
        elif ms and min(ms) > 0.75:
            print('   >> mean 회복률이 높다 — 위치와 무관한 큰 벡터로 설명된다. 해석을 다시 볼 것.')
        print('   ⚠️ 모든 변형이 OOD 다. 절대 수치가 아니라 변형끼리의 비교만 읽을 것.')
    else:
        print('   결과 없음 (위 단계 로그 확인).')

    STATUS['diag_summary'] = summ
    save_status()

stage('A2: diag 요약', _diag_summary)

## 4) 단계 B — seed 학습 (4잡 × 4 GPU, 100k step, ~5.5h)

**이 셀이 몇 시간 블로킹된다.** 이미 100k 인 잡은 skip, 중단된 것(PART)은 resume 한다.
중간에 죽었으면 **노트북을 다시 Run All 하면 이어서 간다.**

In [ ]:
def _train():
    apply_protocol()                      # 어디선가 reload 됐어도 100k 로 되돌린다
    print(f'학습 {v23.STEPS:,} step  |  skip/eval 기준 {cf.CKPT_STEP:,}')
    return cf.run_training_jobs(JOBS, GPUS, prefetch_task=X.TASK)

stage('B: 학습', _train)

# 학습 결과 점검 — 정확히 100k 체크포인트가 있는가
def _has_exact(tag, seed):
    return (v23.train_dir(tag, seed, X.TASK) / 'checkpoints' / str(TRAIN_STEP)).is_dir()

def _ckpt_report():
    rep = {}
    for t, s, task in JOBS:
        d = v23.train_dir(t, s, task) / 'checkpoints'
        steps = sorted(int(x.name) for x in d.iterdir() if x.name.isdigit()) if d.is_dir() else []
        has = _has_exact(t, s)
        rep[f'{t}/seed{s}'] = {'latest': steps[-1] if steps else None, 'has_exact': has}
        print(f'  {t:<22} seed{s}  최신 = {steps[-1] if steps else "없음":>7}  '
              f'{TRAIN_STEP:,} 체크포인트 {"있음" if has else "없음  <- eval 제외"}')
    STATUS['ckpt_after_train'] = rep
    save_status()
    miss = [k for k, v in rep.items() if not v['has_exact']]
    if miss:
        print(f'\n!! {TRAIN_STEP:,} 체크포인트가 없는 잡: {miss} — 이 셀들은 eval 하지 않는다.')

stage('B2: 체크포인트 점검', _ckpt_report)

## 5) 단계 C — eval (seed 별, 50 ep/task = overall 500)

`run_evals` 는 모듈 레벨 `SEED` 를 쓰므로 seed 마다 갈아 끼우며 순차로 돈다. 끝난 셀은 skip 된다.

**정확히 100k 체크포인트가 있는 셀만 eval 한다.** `best_ckpt_dir` 는 그 step 이 없으면
"최근접" 체크포인트로 조용히 대체하기 때문에, 학습이 90k 에서 죽었다면 90k 로 eval 하고도
`100k` 결과처럼 저장될 수 있다. 그걸 막는다.
**끝나면 `SEED` 를 원래대로 되돌린다.**

In [ ]:
EVAL_JOBS = [X._rm_job(v, K, X.MAIN_STRIDE) for v in VARIANTS]
print('eval 셀:', [j['out'] for j in EVAL_JOBS])

def _eval_all():
    apply_protocol()
    print(f'eval 체크포인트 {cf.CKPT_STEP:,} | {X.N_EP} ep/task -> overall {10 * X.N_EP}')
    orig = X.SEED
    try:
        for s in SEEDS:
            print('\n' + '#' * 60 + f'\n# eval seed {s}\n' + '#' * 60)
            jobs = [j for j in EVAL_JOBS if _has_exact(j['src'], s)]
            skipped = [j['out'] for j in EVAL_JOBS if j not in jobs]
            if skipped:
                print(f'   {TRAIN_STEP:,} 체크포인트 없음 -> eval 제외: {skipped}')
                STATUS.setdefault('eval_skipped', {})[str(s)] = skipped
            if not jobs:
                continue
            X.SEED = s
            try:
                X.run_evals(jobs, GPUS)
            except Exception as e:                    # 한 seed 가 죽어도 다음 seed 는 간다
                traceback.print_exc()
                print(f'!! seed {s} eval 실패: {type(e).__name__}: {e}')
                STATUS.setdefault('eval_errors', {})[str(s)] = f'{type(e).__name__}: {e}'
    finally:
        X.SEED = orig
        print('\nSEED 복구 ->', X.SEED)

stage('C: eval', _eval_all)

## 6) 단계 D — 집계 (gap 이 재현되는가)

**이 표가 오늘 밤의 답이다.** seed 0 은 기존 결과(`65.0 / 56.8`)를 그대로 읽는다.

In [ ]:
import statistics as st

def _aggregate():
    rows = []
    orig = X.SEED
    try:
        for s in [0] + SEEDS:
            X.SEED = s
            bi = X._sr(f'rm_bimamba_k{K}_s{X.MAIN_STRIDE}')
            ac = X._sr(f'rm_acm2_k{K}_s{X.MAIN_STRIDE}')
            rows.append({'seed': s, 'bimamba': bi, 'acm2': ac,
                         'gap': (bi - ac) if (bi is not None and ac is not None) else None})
    finally:
        X.SEED = orig

    f = lambda x: f'{x:.1f}' if x is not None else '  --'
    print(f"{'seed':>5} {'BiMamba':>9} {'ACM2':>8} {'gap':>8}")
    print('-' * 33)
    for r in rows:
        print(f"{r['seed']:>5} {f(r['bimamba']):>9} {f(r['acm2']):>8} {f(r['gap']):>8}")

    gaps = [r['gap'] for r in rows if r['gap'] is not None]
    summary = {'rows': rows}
    if len(gaps) >= 2:
        m, sd = st.mean(gaps), st.stdev(gaps)
        summary.update(gap_mean=m, gap_std=sd, n=len(gaps))
        print('-' * 33)
        print(f"{'평균':>5} {'':>9} {'':>8} {m:>8.1f}")
        print(f"{'표준편차':>5} {'':>9} {'':>8} {sd:>8.1f}")
        if sd < 0.3 * abs(m):
            verdict = f'재현된다 (gap {m:.1f} ± {sd:.1f}). 헤드라인 유지.'
        else:
            verdict = f'흔들린다 (gap {m:.1f} ± {sd:.1f}). seed 를 늘리거나 주장을 약하게 할 것.'
        summary['verdict'] = verdict
        print('\n>>', verdict)
    else:
        summary['verdict'] = 'seed 가 하나뿐이다 — 학습/eval 이 덜 끝났는지 확인할 것.'
        print('\n>>', summary['verdict'])

    STATUS['aggregate'] = summary
    save_status()
    (SHARE / f'summary_{STAMP}.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
    return summary

agg = stage('D: 집계', _aggregate)

## 7) 최종 상태

In [ ]:
print('단계별 결과')
print('-' * 50)
for name, s in STATUS['stages'].items():
    mark = 'OK ' if s['ok'] else 'FAIL'
    print(f"  [{mark}] {name:<24} {s['sec']:>7}s" + ('' if s['ok'] else f"   {s.get('error', '')}"))
print()
if agg:
    print('gap 판정 :', agg.get('verdict'))
print('상태 파일:', STATUS_PATH)
print('요약 파일:', SHARE / f'summary_{STAMP}.json')

## 8) 아침에 볼 것

### 먼저 — 브라우저를 닫았다면

셀 출력이 안 남아 있을 수 있다. **파일부터 본다:**

```
outputs/final/share/overnight/status_*.json    단계별 성공/실패, 체크포인트 step, diag3 회복률
outputs/final/share/overnight/summary_*.json   seed 별 성공률과 gap 판정
outputs/final/share/overnight/diag*_all_*_100k.json diag0~3 원본
```

### 결과 읽는 법

| gap 결과 | 뜻 |
|---|---|
| 평균 ~8, 표준편차 < 2.5 | **재현. 헤드라인 유지.** 논문 제일 약한 곳이 사라진다 |
| 표준편차가 평균의 30% 이상 | 흔들린다. seed 를 늘리거나 주장을 약하게 |
| seed 1·2 에서 gap 이 반토막 | **제출 전에 알아서 다행인 경우.** 주장을 다시 잡아야 한다 |

어느 쪽이 나오든 **알고 내는 것과 모르고 내는 것은 다르다.**

### diag3 읽는 법

- **`mean` 회복률 < 25%** → 이득은 거의 전부 위치별 변화에서 온다 (diag2 해석 확정)
- **`mean` 회복률 > 75%** → 위치와 무관한 큰 벡터로 설명된다 (해석 재검토)
- `shuffle` ≈ `random_ortho` → 내용보다 "갈라진다" 가 본질
- 둘 다 `real` 보다 한참 높음 → 어느 위치에 어느 벡터가 가는지도 중요
- ⚠️ 변형은 전부 OOD — **절대 수치가 아니라 변형끼리의 비교만.**

### 중간에 죽었다면

이 노트북을 **다시 Run All** 하면 된다. 100k 인 학습은 skip, 중단된 건 resume,
끝난 eval 은 skip 이다. 단 사전 점검(§2)은 매번 다시 돈다 (몇 초).

### 다음 라운드 후보

1. `acm2_k150` · `acm2_k50` 체크포인트 확인 — diag2(150k)에서는 없다고 나왔다. 단계 A(100k)에서 있으면 표의 55.6 / 45.4 는 100k 것일 수 있고, 둘 다 없으면 표의 셀들이 서로 다른 step 이다
2. E2 `bimamba_scan="random"` — `ecd-bimamba` 머지 + `VARIANTS` 등록 필요
3. K=150 seed 1·2

`JOBS` 와 `EVAL_JOBS` 만 바꾸면 이 노트북을 그대로 재사용한다.